##CELL 1: SETUP & CONFIGURATION

In [0]:
import logging
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import current_timestamp, lit
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_param(param_name: str, default_value: str) -> str:
    try:
        return dbutils.widgets.get(param_name)
    except Exception:
        return default_value

# Pull from orchestrator (variables.yml) or fallback to local defaults
CATALOG = get_param("catalog_name", "main")
SCHEMA = get_param("schema_name", "fixkart_prd")
SOURCE_SYSTEM_NAME = get_param("source_system", "fixkart_core_backend")

# Dynamic Table Paths
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.landing_bookings"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.bronze_bookings"
PRIMARY_KEY = "work_order_id"

logger.info(f"Initialized Bronze Pipeline | Source: {SOURCE_TABLE} | Target: {TARGET_TABLE}")

##CELL 2: BRONZE TRANSFORMATION LOGIC

In [0]:
def enrich_with_bronze_metadata(spark: SparkSession, source_table: str, source_system: str) -> DataFrame:
    """
    Reads raw landing data and appends Bronze-level audit metadata.
    """
    logger.info(f"Reading raw data from {source_table}...")
    
    try:
        df_landing = spark.table(source_table)
        
        df_bronze = (
            df_landing
            .withColumn("ingested_at", current_timestamp())
            .withColumn("source_system", lit(source_system))
        )
        
        logger.info("Successfully applied Bronze audit columns.")
        return df_bronze
        
    except Exception as e:
        logger.error(f"Failed to read or transform landing data: {str(e)}")
        raise

## CELL 3: DELTA LAKE WRITE UTILITIES

In [0]:
def write_bronze_layer(spark: SparkSession, df: DataFrame, target_table: str, join_key: str) -> None:
    """
    Writes the enriched DataFrame to the Bronze Delta table using an idempotent MERGE.
    If the table does not exist, it performs an initial creation.
    """
    logger.info(f"Initiating write to Bronze table '{target_table}'...")
    
    try:
        if spark.catalog.tableExists(target_table):
            logger.info(f"Table {target_table} exists. Performing idempotent MERGE on '{join_key}'...")
            delta_target = DeltaTable.forName(spark, target_table)
            
            (
                delta_target.alias("tgt")
                .merge(
                    df.alias("src"),
                    f"tgt.{join_key} = src.{join_key}"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            logger.info("MERGE operation completed successfully.")
        else:
            logger.info(f"Table {target_table} does not exist. Creating and inserting initial data...")
            (
                df.write.format("delta")
                .option("mergeSchema", "true") 
                .saveAsTable(target_table)
            )
            logger.info("Initial table creation completed successfully.")
            
    except Exception as e:
        logger.error(f"Failed to write to Bronze table {target_table}: {str(e)}")
        raise

## CELL 4: ORCHESTRATION & EXECUTION

In [0]:
if 'spark' in locals() or 'spark' in globals():
    try:
        logger.info("--- Starting Bronze Ingestion Pipeline ---")
        
        # 1. Extract & Transform
        bronze_df = enrich_with_bronze_metadata(
            spark=spark, 
            source_table=SOURCE_TABLE, 
            source_system=SOURCE_SYSTEM_NAME
        )
        
        # 2. Load (Idempotent)
        write_bronze_layer(
            spark=spark,
            df=bronze_df, 
            target_table=TARGET_TABLE,
            join_key=PRIMARY_KEY
        )
        
        logger.info("--- Bronze Ingestion Completed Successfully ---")
        
    except Exception as main_e:
        logger.error("Pipeline execution aborted due to an error.")
else:
    logger.warning("SparkSession not found. Ensure this code is executing in a Databricks environment.")

## CELL 5: DATA VALIDATION

In [0]:

# Order by ingested_at descending to see the freshest data
display(spark.table(TARGET_TABLE).orderBy("ingested_at", ascending=False).limit(20))